In [21]:
import pandas as pd
import numpy as np
import wrds
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 连接WRDS
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\Maoyuqi\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [ ]:
print(" 检查 csmar.trd_mnth 表结构...")

# 检查个股月度交易数据表
try:
    sample_trade = db.raw_sql("SELECT * FROM csmar.trd_mnth LIMIT 5")
    print(f" csmar.trd_mnth 表存在")
    print(f" 列名 ({len(sample_trade.columns)}个):")
    print(f"   {list(sample_trade.columns)}")
    print(f"\n 样本数据:")
    print(sample_trade)
    
    # 检查需要的变量是否存在
    required_trade_vars = ['stkcd', 'trdmnt', 'listdt', 'markettype', 'mclsprc', 'mretwd', 
                          'msmvosd', 'msmvttl', 'mnshrtrd', 'mnvaltrd', 'ndaytrd', 'statco']
    
    print(f"\n 需要的变量匹配情况:")
    for var in required_trade_vars:
        if var in sample_trade.columns:
            print(f"    {var} -> 存在")
        else:
            print(f"    {var} -> 缺失")
            
except Exception as e:
    print(f" 检查表时出错: {e}")

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 定义数据文件夹路径
DATA_DIR = 'data'
RF_DIR = os.path.join(DATA_DIR, 'rf无风险利率')
FINANCIAL_DIR = os.path.join(DATA_DIR, '财务数据')
TRADE_DIR = os.path.join(DATA_DIR, '个股交易数据')
OUTPUT_DIR = '1995-2024月度数据'

# 创建目标文件夹
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_all_data():
    """加载所有原始数据"""
    print(" 加载原始数据...")
    
    # 加载无风险利率数据
    print("  加载无风险利率数据...")
    rf_files = [f for f in os.listdir(RF_DIR) if f.startswith('无风险利率_') and f.endswith('年.csv')]
    rf_data_list = []
    for file in rf_files:
        year = int(file.split('_')[1].split('年')[0])
        df = pd.read_csv(os.path.join(RF_DIR, file))
        df['trdmnt'] = pd.to_datetime(df['trdmnt'])
        df['年份'] = year
        rf_data_list.append(df)
    rf_data = pd.concat(rf_data_list, ignore_index=True).sort_values('trdmnt')
    
    # 加载财务数据（所有年份）
    print("  加载财务数据...")
    
    # 资产负债表
    balance_files = [f for f in os.listdir(FINANCIAL_DIR) if f.startswith('资产负债表_') and f.endswith('年.csv')]
    balance_data_list = []
    for file in balance_files:
        year = int(file.split('_')[1].split('年')[0])
        df = pd.read_csv(os.path.join(FINANCIAL_DIR, file))
        df['accper'] = pd.to_datetime(df['accper'])
        df['年份'] = year
        balance_data_list.append(df)
    balance_data = pd.concat(balance_data_list, ignore_index=True).sort_values(['stkcd', 'accper'])
    
    # 利润表
    income_files = [f for f in os.listdir(FINANCIAL_DIR) if f.startswith('利润表_') and f.endswith('年.csv')]
    income_data_list = []
    for file in income_files:
        year = int(file.split('_')[1].split('年')[0])
        df = pd.read_csv(os.path.join(FINANCIAL_DIR, file))
        df['accper'] = pd.to_datetime(df['accper'])
        df['年份'] = year
        income_data_list.append(df)
    income_data = pd.concat(income_data_list, ignore_index=True).sort_values(['stkcd', 'accper'])
    
    print(f" 数据加载完成:")
    print(f"   - 无风险利率: {len(rf_data)} 条记录, 年份 {rf_data['年份'].min()}-{rf_data['年份'].max()}")
    print(f"   - 资产负债表: {len(balance_data)} 条记录, 年份 {balance_data['年份'].min()}-{balance_data['年份'].max()}")
    print(f"   - 利润表: {len(income_data)} 条记录, 年份 {income_data['年份'].min()}-{income_data['年份'].max()}")
    
    return rf_data, balance_data, income_data

def align_financial_data_corrected(trade_data, financial_data, financial_date_col):
    """财务数据对齐：使用交易日期之前最近的财务数据"""
    aligned_data = []
    
    stock_codes = trade_data['stkcd'].unique()
    total_stocks = len(stock_codes)
    
    print(f"  对齐财务数据，共 {total_stocks} 只股票...")
    
    for i, stock in enumerate(stock_codes):
        if i % 500 == 0 and i > 0:
            print(f"    进度: {i}/{total_stocks} 只股票")
            
        # 获取该股票的交易数据
        stock_trade = trade_data[trade_data['stkcd'] == stock].copy()
        
        # 获取该股票的所有财务数据（所有年份）
        stock_financial = financial_data[financial_data['stkcd'] == stock].sort_values(financial_date_col)
        
        for idx, trade_row in stock_trade.iterrows():
            trade_date = trade_row['trdmnt']
            merged_row = trade_row.copy()
            
            if len(stock_financial) > 0:
                # 找到交易日期之前的所有财务报表
                available_financial = stock_financial[stock_financial[financial_date_col] <= trade_date]
                
                if len(available_financial) > 0:
                    # 取最近的一期财务报表
                    latest_financial = available_financial.iloc[-1]
                    # 添加财务数据列（除了stkcd和日期列）
                    financial_cols = [col for col in latest_financial.index 
                                    if col not in ['stkcd', financial_date_col, '年份']]
                    for col in financial_cols:
                        merged_row[col] = latest_financial[col]
                else:
                    # 如果没有交易日期之前的财务数据，标记为NaN
                    financial_cols = [col for col in financial_data.columns 
                                    if col not in ['stkcd', financial_date_col, '年份']]
                    for col in financial_cols:
                        merged_row[col] = np.nan
            else:
                # 如果该股票没有任何财务数据，标记为NaN
                financial_cols = [col for col in financial_data.columns 
                                if col not in ['stkcd', financial_date_col, '年份']]
                for col in financial_cols:
                    merged_row[col] = np.nan
            
            aligned_data.append(merged_row)
    
    return pd.DataFrame(aligned_data)

def calculate_features_safe(final_data):
    """安全计算特征，处理缺失值"""
    print("  计算特征...")
    
    # 按股票分组计算
    final_data = final_data.sort_values(['stkcd', 'trdmnt'])
    
    def calculate_stock_features_safe(group):
        group = group.sort_values('trdmnt')
        
        try:
            #规模特征
            group['规模'] = np.log(group['msmvttl'].replace(0, np.nan))
        except:
            group['规模'] = np.nan
        
        try:
            #估值特征：账面市值比
            if 'shareholders_equity' in group.columns:
                group['账面市值比'] = group['shareholders_equity'] / group['msmvttl']
            else:
                group['账面市值比'] = np.nan
        except:
            group['账面市值比'] = np.nan
        
        try:
            #盈利能力特征：ROE
            if 'shareholders_equity' in group.columns and 'net_profit_parent' in group.columns:
                # 避免除零
                equity = group['shareholders_equity'].replace(0, np.nan)
                group['净资产收益率'] = group['net_profit_parent'] / equity
            else:
                group['净资产收益率'] = np.nan
        except:
            group['净资产收益率'] = np.nan
        
        try:
            #投资特征：资产增长率
            if 'total_assets' in group.columns:
                group['资产增长率'] = group['total_assets'].pct_change()
            else:
                group['资产增长率'] = np.nan
        except:
            group['资产增长率'] = np.nan
        
        try:
            #动量特征：过去12个月累计收益
            group['动量'] = group['mretwd'].rolling(12, min_periods=6).sum()
        except:
            group['动量'] = np.nan
        
        try:
            #流动性特征：换手率
            denominator = group['msmvosd'] / group['mclsprc'].replace(0, np.nan)
            group['换手率'] = group['mnshrtrd'] / denominator.replace(0, np.nan)
        except:
            group['换手率'] = np.nan
        
        try:
            #流动性特征：Amihud非流动性
            denominator = group['mnvaltrd'].replace(0, np.nan)
            group['非流动性'] = abs(group['mretwd']) / denominator
        except:
            group['非流动性'] = np.nan
        
        try:
            #风险特征：波动率
            group['波动率'] = group['mretwd'].rolling(24, min_periods=12).std()
        except:
            group['波动率'] = np.nan
        
        return group
    
    # 分批处理避免内存问题
    feature_data_list = []
    stock_codes = final_data['stkcd'].unique()
    batch_size = 1000
    
    for i in range(0, len(stock_codes), batch_size):
        batch_codes = stock_codes[i:i + batch_size]
        batch_data = final_data[final_data['stkcd'].isin(batch_codes)]
        
        batch_features = batch_data.groupby('stkcd', group_keys=False).apply(calculate_stock_features_safe)
        feature_data_list.append(batch_features)
        
        if i % 5000 == 0 and i > 0:
            print(f"    进度: {min(i + batch_size, len(stock_codes))}/{len(stock_codes)} 只股票")
    
    return pd.concat(feature_data_list, ignore_index=True)

def process_year_data_corrected(trade_year, rf_data, balance_data, income_data):
    """处理单年数据"""
    print(f"\n 处理 {trade_year} 年数据...")
    
    #加载交易数据
    trade_file = os.path.join(TRADE_DIR, f'全市场个股交易_{trade_year}年.csv')
    if not os.path.exists(trade_file):
        print(f"   交易数据文件不存在: {trade_file}")
        return None
    
    trade_data = pd.read_csv(trade_file)
    trade_data['trdmnt'] = pd.to_datetime(trade_data['trdmnt'])
    print(f"  加载交易数据: {len(trade_data)} 条记录")
    
    #先合并无风险利率
    print("  合并无风险利率...")
    year_rf_data = rf_data[rf_data['年份'] == trade_year].copy()
    merged_data = pd.merge(trade_data, year_rf_data[['trdmnt', 'risk_free_rate']], 
                          on='trdmnt', how='left')
    
    #对齐财务数据（使用交易日期之前最近的财务数据）
    print("  对齐资产负债表...")
    merged_data = align_financial_data_corrected(merged_data, balance_data, 'accper')
    
    print("  对齐利润表...")
    merged_data = align_financial_data_corrected(merged_data, income_data, 'accper')
    
    #计算超额收益率（在财务数据合并后）
    print("  计算超额收益率...")
    merged_data['超额收益率'] = merged_data['mretwd'] - merged_data['risk_free_rate']
    
    #计算特征
    feature_data = calculate_features_safe(merged_data)
    
    #选择最终列
    final_columns = [
        'trdmnt', 'stkcd', 'market_segment', '超额收益率',
        '规模', '账面市值比', '净资产收益率', '资产增长率', 
        '动量', '换手率', '波动率', '非流动性'
    ]
    
    # 只选择存在的列
    available_columns = [col for col in final_columns if col in feature_data.columns]
    final_output = feature_data[available_columns].copy()
    
    # 重命名列
    final_output = final_output.rename(columns={
        'trdmnt': '年月',
        'stkcd': '股票代码', 
        'market_segment': '所属板块'
    })
    
    # 按时间排序
    final_output = final_output.sort_values(['年月', '股票代码'])
    
    return final_output

def analyze_missing_data(data, year):
    """分析数据缺失情况"""
    print(f"\n {year}年数据缺失分析:")
    total_records = len(data)
    
    # 基础信息
    print(f"  总记录数: {total_records}")
    print(f"  股票数量: {data['股票代码'].nunique()}")
    print(f"  时间范围: {data['年月'].min()} 到 {data['年月'].max()}")
    
    # 特征缺失情况
    feature_columns = [col for col in data.columns if col not in ['年月', '股票代码', '所属板块']]
    print(f"\n  特征缺失情况:")
    for col in feature_columns:
        non_null = data[col].notna().sum()
        null_count = data[col].isna().sum()
        coverage = non_null / total_records * 100
        print(f"    {col}: {non_null} 非空, {null_count} 空值 ({coverage:.1f}% 覆盖)")
    
    return data

def main():
    """主处理函数"""
    print(" 开始数据处理和特征计算（data文件夹版本）")
    print("=" * 60)
    
    if not os.path.exists(DATA_DIR):
        print(f" 数据文件夹不存在: {DATA_DIR}")
        print("请确保数据文件夹结构如下:")
        print(f"{DATA_DIR}/")
        print("  ├── rf无风险利率/")
        print("  ├── 财务数据/")
        print("  └── 个股交易数据/")
        return
    
    rf_data, balance_data, income_data = load_all_data()
    
    trade_files = [f for f in os.listdir(TRADE_DIR) if f.startswith('全市场个股交易_') and f.endswith('年.csv')]
    years = sorted([int(f.split('_')[1].split('年')[0]) for f in trade_files])
    
    print(f"\n 可处理年份: {years}")
    
    # 按年份处理数据
    all_processed_data = []
    
    for year in years:
        print(f"\n{'='*50}")
        print(f"处理 {year} 年数据")
        print(f"{'='*50}")
        
        try:
            year_data = process_year_data_corrected(year, rf_data, balance_data, income_data)
            
            if year_data is not None:
                # 分析缺失数据
                year_data = analyze_missing_data(year_data, year)
                
                # 保存结果
                output_file = os.path.join(OUTPUT_DIR, f'全市场特征数据_{year}年.csv')
                year_data.to_csv(output_file, index=False, encoding='utf-8-sig')
                
                all_processed_data.append(year_data)
                print(f" {year}年数据处理完成，保存至: {output_file}")
            
        except Exception as e:
            print(f" {year}年数据处理失败: {e}")
            import traceback
            traceback.print_exc()
    
    # 生成报告
    if all_processed_data:
        print(f"\n{'='*60}")
        print(" 总体处理报告")
        print(f"{'='*60}")
        
        all_data = pd.concat(all_processed_data, ignore_index=True)
        total_years = len(all_processed_data)
        total_records = len(all_data)
        total_stocks = all_data['股票代码'].nunique()
        
        print(f" 所有数据处理完成!")
        print(f" 结果保存在: {OUTPUT_DIR}/")
        print(f" 总体统计:")
        print(f"   - 处理年份: {total_years} 年 ({years[0]}-{years[-1]})")
        print(f"   - 总记录数: {total_records}")
        print(f"   - 总股票数: {total_stocks}")
        print(f"   - 时间范围: {all_data['年月'].min()} 到 {all_data['年月'].max()}")
        
        # 总体缺失分析
        print(f"\n 总体缺失分析:")
        feature_columns = [col for col in all_data.columns if col not in ['年月', '股票代码', '所属板块']]
        for col in feature_columns:
            non_null = all_data[col].notna().sum()
            coverage = non_null / total_records * 100
            print(f"   {col}: {coverage:.1f}% 覆盖")

# 运行主程序
if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os

base_path = "1995-2024月度数据"
key_features = ["规模", "账面市值比", "净资产收益率", "资产增长率"]
years = range(1995, 2025)

for year in years:
    print(f"\n===============================")
    print(f" 处理 {year} 年")
    print("===============================")

    file_path = os.path.join(base_path, f"全市场特征数据_{year}年.csv")
    df = pd.read_csv(file_path)
    df = df.sort_values(["年月", "股票代码"])

    # 按股票计算缺失率
    missing_rate = (
        df.groupby("股票代码")[key_features]
        .apply(lambda x: x.isna().mean())
    )

    # 标记缺失率是否 > 20%
    stocks_to_drop = missing_rate[
        (missing_rate > 0.20).any(axis=1)
    ].index.tolist()

    print(f" 删除股票数量（缺失率>20%）: {len(stocks_to_drop)}")

    # 删除当年的这些股票
    df = df[~df["股票代码"].isin(stocks_to_drop)]

    #对剩余股票进行前向填充 + 回填
    df[key_features] = (
        df.groupby("股票代码")[key_features]
        .apply(lambda x: x.ffill().bfill())
        .reset_index(level=0, drop=True)
    )

    # 检查100%覆盖率
    coverage = df[key_features].notna().mean() * 100
    print(" 填充后覆盖率：")
    for k, v in coverage.items():
        print(f"   {k}: {v:.2f}%")

    # 保存 
    out_path = os.path.join(base_path, f"清洗选中特征后-全市场特征数据_{year}年.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f" 已保存到: {out_path}")


In [22]:
# 关闭数据库连接
db.close()
print('数据库连接已关闭')

数据库连接已关闭
